# 🚀 OpenClaw → Wan 2.2 — ONE CLICK

**Cuma 1 cell.** Pilih GPU di Colab, lalu tekan ▶️ sekali. Sisanya otomatis.

Target: iPhone → OpenClaw → ComfyUI → Wan 2.2 → video.


In [ ]:
# ===== ONE CLICK SETUP =====
import subprocess,sys,os,time,requests,re
print('1/6 Cek GPU...')
subprocess.run(['nvidia-smi'],check=False)
print('2/6 Install ComfyUI + CUDA PyTorch...')
subprocess.run(['git','clone','--depth','1','https://github.com/comfyanonymous/ComfyUI.git','/content/ComfyUI'],check=False)
os.chdir('/content/ComfyUI')
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','--force-reinstall','--no-cache-dir','torch==2.10.0','--index-url','https://download.pytorch.org/whl/cu126'],check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','requests','websocket-client'],check=False)
import torch
print('PyTorch:',torch.__version__,'CUDA:',torch.version.cuda,'AVAILABLE:',torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('GPU belum aktif. Colab: Runtime → Change runtime type → GPU, lalu jalankan cell ini lagi.')
print('GPU:',torch.cuda.get_device_name(0))
print('3/6 Download model Wan 2.2...')
os.makedirs('models/diffusion_models',exist_ok=True);os.makedirs('models/text_encoders',exist_ok=True);os.makedirs('models/vae',exist_ok=True)
files=[('models/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors','https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors'),('models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors','https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors'),('models/vae/wan2.2_vae.safetensors','https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan2.2_vae.safetensors')]
for path,url in files:
    if not os.path.exists(path) or os.path.getsize(path)<1000000:
        subprocess.run(['wget','-q','--show-progress','-O',path,url],check=True)
print('4/6 Start ComfyUI...')
log=open('/content/comfyui.log','w')
comfy=subprocess.Popen([sys.executable,'main.py','--listen','0.0.0.0','--port','8188','--enable-cors-header','*'],stdout=log,stderr=subprocess.STDOUT,text=True)
ready=False
for _ in range(120):
    try:
        if requests.get('http://127.0.0.1:8188/system_stats',timeout=2).ok:
            ready=True;break
    except: pass
    if comfy.poll() is not None: break
    time.sleep(1)
log.close()
if not ready:
    print(open('/content/comfyui.log',errors='ignore').read()[-10000:])
    raise RuntimeError('ComfyUI gagal start.')
print('COMFYUI READY ✅')
print('5/6 Buat URL publik...')
subprocess.run(['wget','-q','https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64','-O','/content/cloudflared'],check=True)
os.chmod('/content/cloudflared',0o755)
cf_log=open('/content/cloudflared.log','w')
cf=subprocess.Popen(['/content/cloudflared','tunnel','--no-autoupdate','--url','http://127.0.0.1:8188'],stdout=cf_log,stderr=subprocess.STDOUT,text=True)
url=None
for _ in range(90):
    cf_log.flush()
    txt=open('/content/cloudflared.log',errors='ignore').read()
    m=re.search(r'https://[A-Za-z0-9.-]+\.trycloudflare\.com',txt)
    if m: url=m.group(0);break
    time.sleep(1)
cf_log.close()
if not url: raise RuntimeError('Cloudflare gagal. Log: '+open('/content/cloudflared.log',errors='ignore').read()[-5000:])
open('/content/COMFYUI_BASE_URL.txt','w').write(url)
print('6/6 SELESAI 🎉')
print('COMFYUI_BASE_URL =',url)
print('Simpan URL ini untuk OpenClaw. Jangan tutup runtime Colab.')
